In [1]:
"""Combine HCDP monthly station rainfall CSVs into a single long-format CSV.

Each source file is wide format: one row per station, one column per day
(e.g. X1990.01.01). This melts those day columns into rows and stacks every
monthly file into one dataset.

Scope: filtered to Oahu stations only (Island == "OA"), since Kaimaemae only
models Oahu beaches. Full date range (1990-2026) is preserved. Rainfall values
are left raw (no gap filling) for the downstream cleaning step.

Output: resources/raw_data/rainfall_data.csv
"""

import glob
import os
import re

import pandas as pd

# Notebook lives in backend/data_scripts/ -> go up two levels to project root.
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
RAW_DATA = os.path.join(PROJECT_ROOT, "resources", "raw_data")
STATION_DATA = os.path.join(
    RAW_DATA, "HCDP_rainfall_data", "rainfall", "new", "day",
    "statewide", "partial", "station_data",
)
OUTPUT = os.path.join(RAW_DATA, "rainfall_data.csv")

# Station metadata columns we keep, mapped to clean output names.
META = {
    "SKN": "station_id",
    "Station.Name": "station_name",
    "Island": "island",
    "LAT": "lat",
    "LON": "lon",
    "ELEV.m.": "elevation_m",
}

DAY_COL = re.compile(r"^X(\d{4})\.(\d{2})\.(\d{2})$")


def melt_file(path):
    df = pd.read_csv(path, dtype=str)

    # Keep only Oahu stations.
    df = df[df["Island"] == "OA"]
    if df.empty:
        return None

    day_cols = [c for c in df.columns if DAY_COL.match(c)]
    long = df.melt(
        id_vars=list(META.keys()),
        value_vars=day_cols,
        var_name="day_col",
        value_name="rainfall_mm",
    )

    # Convert X1990.01.01 -> 1990-01-01
    long["date"] = long["day_col"].str.replace(DAY_COL, r"\1-\2-\3", regex=True)
    long = long.drop(columns=["day_col"]).rename(columns=META)
    return long


files = sorted(glob.glob(os.path.join(STATION_DATA, "*", "*", "*.csv")))
print(f"Found {len(files)} monthly files")

frames = []
for i, path in enumerate(files, 1):
    part = melt_file(path)
    if part is not None:
        frames.append(part)
    if i % 50 == 0:
        print(f"  processed {i}/{len(files)} files")

combined = pd.concat(frames, ignore_index=True)
combined = combined[
    ["date", "station_id", "station_name", "island",
     "lat", "lon", "elevation_m", "rainfall_mm"]
]
combined = combined.sort_values(["station_id", "date"]).reset_index(drop=True)

combined.to_csv(OUTPUT, index=False)
print(f"\nWrote {len(combined):,} rows to {OUTPUT}")
print(f"Stations: {combined['station_id'].nunique()}")
print(f"Date range: {combined['date'].min()} to {combined['date'].max()}")
combined.head()


Found 436 monthly files
  processed 50/436 files
  processed 100/436 files
  processed 150/436 files
  processed 200/436 files
  processed 250/436 files
  processed 300/436 files
  processed 350/436 files
  processed 400/436 files

Wrote 1,482,457 rows to c:\Users\ralph\projects\kaimaemae\resources\raw_data\rainfall_data.csv
Stations: 175
Date range: 1990-01-01 to 2026-04-30


,date,station_id,station_name,island,lat,lon,elevation_m,rainfall_mm
0,1990-01-01,702.2,HONOLULU OBSERV 702.2,OA,21.313926,-157.999529,2,1.99875
1,1990-01-02,702.2,HONOLULU OBSERV 702.2,OA,21.313926,-157.999529,2,1.53425
2,1990-01-03,702.2,HONOLULU OBSERV 702.2,OA,21.313926,-157.999529,2,0.267
3,1990-01-04,702.2,HONOLULU OBSERV 702.2,OA,21.313926,-157.999529,2,0
4,1990-01-05,702.2,HONOLULU OBSERV 702.2,OA,21.313926,-157.999529,2,0
